# Ouroboros Quickstart (Gemini 2.5 & iFlow Edition)

A self-modifying AI agent that writes its own code and evolves autonomously.

**Before running:**

1. [Fork the repository](https://github.com/joi-lab/ouroboros/fork) on GitHub
2. Add your API keys in the **Secrets** sidebar (key icon on the left):
   - `IFLOW_API_KEY` — required (from https://platform.iflow.cn)
   - `TELEGRAM_BOT_TOKEN` — required ([@BotFather](https://t.me/BotFather))
   - `TOTAL_BUDGET` — required, spending limit in USD (e.g. `50`)
   - `GITHUB_TOKEN` — required ([github.com/settings/tokens](https://github.com/settings/tokens), `repo` scope)
   - `GOOGLE_API_KEY` — required (for Gemini Pro)
3. **Check `GITHUB_USER`** in the cell below
4. Run the cell (Shift+Enter)
5. Open your Telegram bot and send Любое сообщение — you become the owner

In [ ]:
import os
import shutil
from google.colab import userdata

# ⚠️ Конфигурация для Aisrefot-Reed
CFG = {
    "GITHUB_USER": "Aisrefot-Reed",
    "GITHUB_REPO": "ouroboros",
    
    # ОСНОВНОЙ МОЗГ: Gemini 2.5 Pro (Самая стабильная и умная в API)
    "OUROBOROS_MODEL": "google/gemini-2.5-pro", 
    
    # КОДИНГ: Qwen3-Coder-Plus (через iFlow)
    "OUROBOROS_MODEL_CODE": "Qwen3-Coder-Plus",
    
    # ЛЕГКИЕ ЗАДАЧИ: Gemini 2.5 Flash-Lite
    "OUROBOROS_MODEL_LIGHT": "google/gemini-2.5-flash-lite",
    
    # ВЕБ-РЕСЕРЧ: Gemini Thinking
    "OUROBOROS_WEBSEARCH_MODEL": "google/gemini-2.0-flash-thinking-exp",
    
    "OUROBOROS_MAX_WORKERS": "5",
    "OUROBOROS_MAX_ROUNDS": "200",
    "OUROBOROS_BG_BUDGET_PCT": "15",
}

# 1. Автоматическая загрузка секретов из панели Colab (Secrets)
SECRETS = [
    "IFLOW_API_KEY", "TELEGRAM_BOT_TOKEN", "TOTAL_BUDGET", 
    "GITHUB_TOKEN", "GOOGLE_API_KEY", "OPENAI_API_KEY", "TAVILY_API_KEY"
]

print("--- Загрузка секретов ---")
for s in SECRETS:
    try:
        val = userdata.get(s)
        os.environ[s] = str(val)
        print(f"✅ {s} загружен")
    except:
        if s in ["IFLOW_API_KEY", "TELEGRAM_BOT_TOKEN", "TOTAL_BUDGET", "GITHUB_TOKEN", "GOOGLE_API_KEY"]:
            print(f"❌ ПРЕДУПРЕЖДЕНИЕ: Секрет {s} не найден!")
        else:
            print(f"⚠️ {s} не задан (опционально)")

# 2. Применяем конфигурацию
for k, v in CFG.items():
    if k not in os.environ:
        os.environ[k] = str(v)

# 3. Подготовка репозитория
%cd /content
if os.path.exists("/content/ouroboros_repo"):
    print("Очистка старого репозитория...")
    shutil.rmtree("/content/ouroboros_repo")

repo_url = f"https://github.com/{os.environ['GITHUB_USER']}/{os.environ['GITHUB_REPO']}.git"
print(f"Клонирование {repo_url}...")
!git clone {repo_url} /content/ouroboros_repo
%cd /content/ouroboros_repo

# 4. Установка зависимостей и браузера
print("Установка системных компонентов...")
!pip install -q -r requirements.txt
!pip install -q google-generativeai playwright-stealth
!playwright install chromium
!playwright install-deps chromium

# 5. Запуск
print("Запуск Ouroboros...")
%run colab_bootstrap_shim.py